# Profiling Silver

Este notebook perfila la capa Silver. Silver contiene datasets limpios y consolidados, no facts ni dimensiones finales.

Objetivo:
- Inventariar datasets curados.
- Revisar tipos normalizados.
- Medir nulos, duplicados y trazabilidad.
- Revisar cuarentenas cuando existan.
- Generar reportes HTML separados en `reports/profiling/silver`.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

ROOT = Path('/home/jovyan/work')
SILVER = ROOT / 'data' / 'silver'
QUARANTINE = SILVER / '_quarantine'
spark = SparkSession.builder.appName('silver-profiling-notebook').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
tables = sorted(p for p in SILVER.iterdir() if p.is_dir() and p.name != '_quarantine' and any(p.rglob('*.parquet')))
[(p.name, str(p)) for p in tables]

In [ ]:
summary = []
for path in tables:
    df = spark.read.parquet(str(path))
    trace_cols = [c for c in df.columns if c.startswith('_bronze') or c.startswith('_silver')]
    summary.append((path.name, df.count(), len(df.columns), len(trace_cols)))
spark.createDataFrame(summary, ['dataset_silver', 'registros', 'columnas', 'columnas_trazabilidad']).orderBy('dataset_silver').show(200, truncate=False)

## Validaciones Silver

Silver debe demostrar limpieza, estandarizacion, tipos correctos, tratamiento de nulos, deduplicacion y trazabilidad hacia Bronze.

In [ ]:
for path in tables:
    df = spark.read.parquet(str(path))
    print('\n===', path.name, '===')
    print('registros:', df.count(), 'columnas:', len(df.columns))
    print('duplicados fila completa:', df.count() - df.dropDuplicates().count())
    df.printSchema()
    cols = df.columns[:25]
    if cols:
        df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in cols]).show(truncate=False)
    df.limit(5).show(truncate=False)

In [ ]:
if QUARANTINE.exists():
    quarantine_tables = sorted(p for p in QUARANTINE.iterdir() if p.is_dir() and any(p.rglob('*.parquet')))
    for path in quarantine_tables:
        df = spark.read.parquet(str(path))
        print('\nCUARENTENA:', path.name, 'registros:', df.count())
        df.limit(10).show(truncate=False)
else:
    print('No existe carpeta de cuarentena Silver')

In [ ]:
# Generar HTML de profiling para Bronze, Silver y Gold.
# La salida Silver queda en reports/profiling/silver/index.html
%run ../../scripts/profile_medallion_layers.py